<a href="https://colab.research.google.com/github/PlaZMaD/MLDM-2026/blob/main/02-linear-regression/LinearRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Linear Regression

All the imports live in a single cell, and every random draw in this notebook goes through
a seeded generator. Re-running any cell gives exactly the same picture, so the plots you see
are the plots the seminar talks about.

Each section also uses its own names for the data (`X_toy`, `X_poly`, `X_lin`) instead of
reusing `X` and `y`. Running the cells out of order then fails loudly instead of silently
plotting the wrong thing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import trange, tqdm

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

SEED = 42

## y = kx + b

Let's start with a toy 1D problem, where the true dependence is
$$y=k\cdot x+b+\text{noise}$$

In [ ]:
def linear_function(x):
    return 0.33 * x + 8.3

def gen_dataset(N, func, lims=(-1., 1.), noise_lvl=0.2, seed=SEED):
    rng = np.random.default_rng(seed)
    x = rng.uniform(*lims, size=N)
    y = func(x) + noise_lvl * rng.normal(size=x.shape)
    return x[:, None], y


def plot_fit(X, y, func, model=None, lims=(-1., 1.), label='prediction'):
    """Data + true dependence + (optionally) a model prediction.

    Used everywhere below instead of copy-pasting the same four lines.
    """
    x = np.linspace(*lims, 201)
    plt.plot(x, func(x), c='black', label='true function')
    plt.scatter(X, y, s=25, label='data')
    if model is not None:
        plt.plot(x, model.predict(x[:, None]), label=label)
        plt.ylim(y.min() - 0.5, y.max() + 0.5)
    plt.legend()


X_toy, y_toy = gen_dataset(50, linear_function)
plot_fit(X_toy, y_toy, linear_function);

### `LinearRegression` from `sklearn`

In [ ]:
# The following class implements the analytical solution for
# linear regression with the MSE loss
model_toy = LinearRegression().fit(X_toy, y_toy)

plot_fit(X_toy, y_toy, linear_function, model_toy)
print(model_toy.coef_, model_toy.intercept_)

### Sidenote: making contour plots

In [ ]:
### Sidenote: making contour plots (level maps)

plt.contourf(
    [[0., 1., 2.], # matrix of X
     [0., 1., 2.],
     [0., 1., 2.]],
    [[ 0.,  0.,  0.], # matrix of Y
     [10., 10., 10.],
     [20., 20., 20.]],
    [[-1., 0., -1.], # matrix of Z
     [ 0., 1.,  0.],
     [-1., 0., -1.]]
);

In [ ]:
### 2D matrices of X and Y (as above) can be
### created from 1D vectors using np.meshgrid:

for i in np.meshgrid([0., 1., 2], [0., 10., 20.]):
    print(i)

### MSE as a function of model parameters

Let's see what MSE looks like as a function of model parameters **(2 points)**:

In [ ]:
# Creating a grid of model parameter values:
ww, bb = np.meshgrid(
    np.linspace(-10., 10., 50),
    np.linspace(-5., 15., 50)
)

# Your turn: calculate the map of MSE values on the grid defined above, i.e.
# for each (w, b) in (ww, bb) calculate MSE for the model y = w * x + b
# on the (X_toy, y_toy) dataset. Avoid using loops.
MSE_map = <YOUR CODE>

# Automatic checks
assert MSE_map.shape == ww.shape
for i in [0, -1]:
    for j in [0, -1]:
        assert np.isclose(
            MSE_map[i, j],
            ((ww[i, j] * X_toy.ravel() + bb[i, j] - y_toy)**2).mean()
        ), f'assert failed for point {i, j}'

# Plotting:
plt.figure(figsize=(6, 5), dpi=100)
plt.colorbar(plt.contourf(ww, bb, MSE_map, levels=30))
plt.xlabel('$w$'); plt.ylabel('$b$')
# the star is the analytical solution found by sklearn above
plt.scatter(model_toy.coef_, model_toy.intercept_, marker='*', s=150, c='orange');

## Polynomial fit

Now let's take some arbitrary function:

In [ ]:
def true_function(x):
    return np.sin(3 * x + 0.8) + np.sin(1. / (x + 1.23))

Obviously, we won't get a good fit with an ordinary linear regression:

In [ ]:
# A separate dataset with its own name and its own seed:
# from here on the polynomial part of the notebook works with (X_poly, y_poly).
X_poly, y_poly = gen_dataset(25, true_function, seed=SEED + 1)

plot_fit(X_poly, y_poly, true_function, LinearRegression().fit(X_poly, y_poly));

### `PolynomialFeatures` and pipelines

Even though our design matrix has only one column:
$$X=
\begin{pmatrix}
x_1 \\
x_2 \\
\vdots \\
x_N
\end{pmatrix},
$$
we can expand it with powers of $x$ to fit a polynomial:
$$X'=
\begin{pmatrix}
x_1 & (x_1)^2 & \ldots & (x_1)^k \\
x_2 & (x_2)^2 & \ldots & (x_2)^k \\
\vdots & \vdots & \ddots & \vdots \\
x_N & (x_N)^2 & \ldots & (x_N)^k
\end{pmatrix},
$$

such that:

$$\frac{1}{N}\left\Vert X'\cdot w - y\right\Vert^2\to \underset{w}{\text{min}}$$

This functionality is implemented in `sklearn.preprocessing.PolynomialFeatures`:

In [ ]:
poly_demo = PolynomialFeatures(3)
poly_demo.fit_transform(np.arange(5)[:, None])

One can combine `PolynomialFeatures` (and any other transformers) along with the model into a single pipeline (`sklearn.pipeline.make_pipeline`):

In [ ]:
# The first parameter is the power of expansion. Try playing around with it.
poly_expand = PolynomialFeatures(5, include_bias=False)
linear_model = LinearRegression()
model_poly = make_pipeline(poly_expand, linear_model)

model_poly.fit(X_poly, y_poly)

plot_fit(X_poly, y_poly, true_function, model_poly);

Now we want to plot 2D projections of MSE as a function of model parameters **(2 points)**.

The parameter vector and the expanded design matrix are built here, in a cell of their own:
the gradient descent section below needs them too, and it should not break if you get stuck
on the task in the next cell.

In [ ]:
# Combine the weights and the bias into a single parameter vector
solution = np.concatenate([linear_model.coef_, [linear_model.intercept_]])

# Calculate the power expansion of the features (bias column goes last,
# to match the layout of `solution`)
X_expanded = np.concatenate([
    poly_expand.transform(X_poly), np.ones(shape=(len(X_poly), 1))
], axis=1)

print(solution.shape, X_expanded.shape)

In [ ]:
# We'll plot a large matrix of plots, so let's create
# a 16 by 16 inch canvas
plt.figure(figsize=(16, 16))

# We'll loop over all pairs of weights
i_img = 0
for dim1 in trange(len(solution)):
    for dim2 in range(len(solution)):
        i_img += 1
        # Skip the diagonal
        if dim1 == dim2: continue

        # Create the grid of parameter values
        ww1, ww2 = np.meshgrid(
            np.linspace(solution[dim1] - 1000., solution[dim1] + 1000., 50),
            np.linspace(solution[dim2] - 1000., solution[dim2] + 1000., 50),
        )

        # `param_grid` is a 3D array of parameter values of the
        # following shape: (len(solution), ww1.shape[0], ww1.shape[1]),
        # such that `param_grid[i, :, :]` equals to:
        #     `ww1` if `i` equals `dim1`;
        #     `ww2` if `i` equals `dim2`;
        #     `solution[i]` otherwise.
        param_grid = np.empty(shape=(len(solution),) + ww1.shape, dtype=float)
        param_grid[:] = solution[:, None, None]
        param_grid[dim1] = ww1
        param_grid[dim2] = ww2

        # Automatic checks
        assert param_grid.shape == (len(solution),) + ww1.shape
        assert np.allclose(param_grid[dim1], ww1)
        assert np.allclose(param_grid[dim2], ww2)
        assert all(
            np.allclose(param_grid[i], solution[i])
            for i in range(len(solution)) if i not in (dim1, dim2)
        )

        # Your turn! Now it's time to calculate the MSE map, i.e. for each grid
        # element (i, j), you want `MSE_map[i, j]` to be equal to the MSE
        # for the model defined by parameters `param_grid[:, i, j]`.
        MSE_map = <YOUR CODE>

        # Automatic checks
        assert MSE_map.shape == ww1.shape
        for i in [0, -1]:
            for j in [0, -1]:
                assert np.isclose(
                    ((X_expanded @ param_grid[:, i, j] - y_poly)**2).mean(),
                    MSE_map[i, j]
                ), f'Check failed for point {i, j}'

        plt.subplot(len(solution), len(solution), i_img)
        plt.axis('off')

        plt.contourf(ww1, ww2, MSE_map, levels=10);
        plt.scatter(solution[dim1], solution[dim2], marker='*', s=30, c='orange')

Note the relation between the amount of overfitting and correlation between parameters.

## Gradient descent

Let's look at MSE as a function of parameters:
$$\text{MSE}(w)=\frac{1}{N}\left\Vert X'\cdot w - y\right\Vert^2$$

Instead of minimizing it analytically, we can use numeric optimization with gradient descent. I.e. do the following procedure iteratively:
$$w\leftarrow w-\alpha\cdot\frac{\partial\text{MSE}(w)}{\partial w},$$
for some constant *learning rate* $\alpha$.

For the task below you'll need to derive the analytical formula for $\frac{\partial\text{MSE}(w)}{\partial w}$. **Note, that $w$ is a vector!** If not sure how to do it, check out the [matrix calculus cheat sheet](https://en.wikipedia.org/wiki/Matrix_calculus#Identities).

Your formula is checked against a finite-difference approximation of the same gradient, so a
missing factor of 2 or a wrong normalization is caught immediately, rather than quietly
rescaling your learning rate.

When done **(2 points)**, play around with the power of the polynomial expansion, learning rate and the number of gradient descent steps.

In [ ]:
def mse_loss(w, X=X_expanded, y=y_poly):
    return ((X @ w - y)**2).mean()


def mse_grad(w, X=X_expanded, y=y_poly):
    # Your turn: return the gradient of `mse_loss` with respect to w
    return <YOUR CODE>


def numeric_grad(f, w, eps=1e-6):
    """Central finite difference: (f(w + eps) - f(w - eps)) / (2 * eps) per coordinate."""
    return np.array([
        (f(w + eps * e) - f(w - eps * e)) / (2 * eps)
        for e in np.eye(len(w))
    ])


# Automatic checks: the analytical gradient must match the numerical one
w_check = np.random.default_rng(SEED).normal(size=X_expanded.shape[1]) * 0.1
assert np.shape(mse_grad(w_check)) == w_check.shape
assert np.allclose(mse_grad(w_check), numeric_grad(mse_loss, w_check), rtol=1e-4, atol=1e-5), (
    'analytical and numerical gradients disagree -- check the factor of 2 and the 1/N'
)
print('gradient OK')

In [ ]:
# Initialize the model parameters with zeros
w = np.zeros(dtype=float, shape=X_expanded.shape[1])

loss_values = []  # a list to keep track of how the loss value changes
learning_rate = 0.1

# Training loop
for _ in trange(1000):
    # Gradient descent step
    w -= learning_rate * mse_grad(w)

    # Calculate and record the new loss value
    loss_values.append(mse_loss(w))

# Plotting the evolution of loss values
plt.plot(loss_values)
plt.xlabel('iteration'); plt.ylabel('MSE')
plt.show()

print(f'MSE: gradient descent = {mse_loss(w):.5f}, '
      f'analytical solution = {mse_loss(solution):.5f}')

# Plotting the solution
x = np.linspace(-1, 1, 101)
x_expanded = np.concatenate([
    poly_expand.transform(x[:, None]),
    np.ones(shape=(len(x), 1))
], axis=1)
plt.plot(x, true_function(x), c='black', label='true function')
plt.scatter(X_poly, y_poly, s=25)
plt.plot(x, x_expanded @ w, label='prediction')
plt.ylim(y_poly.min() - 0.5, y_poly.max() + 0.5)
plt.legend();

Did you notice that numeric solution is less prone to overfitting? Some intuition for that can be found in this post: https://distill.pub/2017/momentum/ (though not explicitly).

The MSE printed above makes it concrete: gradient descent stops at a *worse* training loss
than the analytical solution, and that is exactly why its curve is smoother.

## Here comes the overfitting

What is going to happen if we increase the number of powers we use?

In [ ]:
# Define the set of powers to be used generating the features
set_of_powers = [1, 5, 15, 25]

# Plotting the ground truth (same (X_poly, y_poly) as above -- the data is seeded,
# so this plot is directly comparable with the fits we have already seen)
plt.figure(figsize=(15, 6))
x = np.linspace(-1, 1, 200)
plt.plot(x, true_function(x), c='black', label='true function')
plt.scatter(X_poly, y_poly, marker='*', s=100)
plt.ylim(-2, 2)

# Adding plots for every single value in the set of powers using the pipeline
for d in set_of_powers:
    model_d = make_pipeline(PolynomialFeatures(d, include_bias=False), LinearRegression())
    model_d.fit(X_poly, y_poly)
    plt.plot(x, model_d.predict(x[:, None]), linewidth=2, label="$d=%d$" % d)

plt.legend(loc=1)
plt.title('Polynomial fits of increasing degree')
plt.show()

Eyeballing the curves is not an argument, though. Let's measure the effect: fit on a train
split and look at the MSE on points the model has never seen.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_poly, y_poly, test_size=0.4, random_state=SEED
)

degrees = range(1, 16)
train_mse, test_mse = [], []
for d in degrees:
    model_d = make_pipeline(
        PolynomialFeatures(d, include_bias=False), LinearRegression()
    ).fit(X_tr, y_tr)
    train_mse.append(mean_squared_error(y_tr, model_d.predict(X_tr)))
    test_mse.append(mean_squared_error(y_te, model_d.predict(X_te)))

plt.figure(figsize=(7, 4), dpi=100)
plt.semilogy(list(degrees), train_mse, marker='o', label='train')
plt.semilogy(list(degrees), test_mse, marker='o', label='test')
plt.xlabel('polynomial degree'); plt.ylabel('MSE')
plt.legend(); plt.grid(alpha=.3)

best = list(degrees)[int(np.argmin(test_mse))]
print(f'best degree by test MSE: {best}')

Using `sklearn` LinearRegression, it is possible to fit every single point, but it is not the objective we try to achieve: the train curve keeps going down while the test curve turns around.

Two follow-ups worth trying here:

* at $d=25$ with 25 points the design matrix is effectively singular, and the fitted coefficients blow up -- print `np.abs(model_d[-1].coef_).max()` for a few degrees;
* `Ridge(alpha=...)` instead of `LinearRegression` keeps the high-degree fit sane. That is the bridge to the regularization topic.

## Bonus. Custom LinearRegression class

Based on [OML](https://github.com/girafe-ai/ml-mipt) open materials.

Let's wrap our previous Gradient Descent experiments into LR class, using `sklearn` standard interfaces to implement it **(3 points)**. You can find [the base classes](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.base) for all of the most common problems. So, we need to inherit base class and implement main stages of regression pipeline methods:
* hyperparameter initialization - constructor
* parameters training on known objects - fit method
* target estimation for unknown objects - predict method

Two conventions worth following literally, because `sklearn` relies on them:

* everything learned during `fit` gets a **trailing underscore** (`w_`, `w_history_`). `Pipeline.predict` calls `check_is_fitted`, which looks exactly for such attributes -- store the weights as `self.w` and the pipeline at the bottom of this notebook will raise `NotFittedError` even though your model is perfectly fitted;
* `__init__` only assigns its arguments to attributes of the same name -- no lists, no computation, no validation. Otherwise `get_params` / `clone` break, and mutable state created in the constructor survives across `fit` calls (a history list would accumulate two runs and the trajectory plot below would show both).

In [ ]:
class LinearRegressionSGD(BaseEstimator, RegressorMixin):  # inherit base class
    def __init__(self,
                 batch_size: int = 10,
                 lr: float = 1e-2,
                 n_iters: int = 10,
                 random_state: int = None) -> None:
        # You can read more about Annotations here: https://www.python.org/dev/peps/pep-3107/
        # Only assignments here -- see the note above.
        self.batch_size = batch_size
        self.lr = lr
        self.n_iters = n_iters
        self.random_state = random_state

    def fit(self, X, y):
        rng = np.random.default_rng(self.random_state)
        n_objects = len(X)

        # initialize the weights
        w = <YOUR CODE>

        # We are going to use the history plotting the trajectory.
        # It belongs to `fit`, so that a second `fit` starts from scratch.
        history = [w.copy()]

        for i in range(self.n_iters):
            # Sample random indices of objects to be used during the step
            sample_indices = <YOUR CODE>

            # Make the step
            w -= <YOUR CODE>
            history.append(w.copy())

        # Trailing underscores: this is what `check_is_fitted` looks for
        self.w_ = w
        self.w_history_ = np.array(history)
        return self

    def predict(self, X):
        # Return the results of your model applied to X
        return <YOUR CODE>

Now we can have a look on the weights' trajectory through the training process.

In [ ]:
n_features = 2
n_objects = 300
batch_size = 10
num_steps = 15

# Seed and data generation live in the same cell on purpose:
# re-running this cell alone must not change the dataset.
rng = np.random.default_rng(43)

# Let it be the *true* weights vector
w_true = rng.normal(size=(n_features))
X_lin = rng.uniform(-5, 5, (n_objects, n_features))

# Let's make different scales of features
X_lin *= (np.arange(n_features) * 2 + 1)[np.newaxis, :]

# Here you define the *true* target value
y_lin = X_lin.dot(w_true) + rng.normal(0, 1, n_objects)

# Split the data
x_train, x_test, y_train, y_test = train_test_split(
    X_lin, y_lin, random_state=SEED
)

# Fit the model of our custom class
own_lr = LinearRegressionSGD(
    batch_size=batch_size, n_iters=num_steps, random_state=SEED
).fit(x_train, y_train)

# Get the weights
w_list = own_lr.w_history_
print('test R^2:', own_lr.score(x_test, y_test))

In [ ]:
# Compute level set -- on the same data the model was fitted on (the train part),
# otherwise the trajectory and the contours would refer to different loss functions
A, B = np.meshgrid(np.linspace(-2, 2, 100), np.linspace(-2, 2, 100))

W_grid = np.stack([A, B])                            # (2, 100, 100)
preds = np.tensordot(x_train, W_grid, axes=(1, 0))   # (n_train, 100, 100)
levels = ((preds - y_train[:, None, None])**2).mean(axis=0)

plt.figure(figsize=(13, 9))
plt.title('LR weights GD trajectory')
plt.xlabel('$w_1$')
plt.ylabel('$w_2$')
plt.xlim(w_list[:, 0].min() - 0.1, w_list[:, 0].max() + 0.1)
plt.ylim(w_list[:, 1].min() - 0.1, w_list[:, 1].max() + 0.1)
plt.gca().set_aspect('equal')

# Visualize the level set
CS = plt.contour(A, B, levels, levels=np.logspace(0, 1.8, num=10), cmap=plt.cm.rainbow_r)
CB = plt.colorbar(CS, shrink=0.7)

# Visualize the trajectory
plt.scatter(w_true[0], w_true[1], c='r', marker='*', s=200)
plt.scatter(w_list[:, 0], w_list[:, 1])
plt.plot(w_list[:, 0], w_list[:, 1])

plt.show()

Self-check. `cross_val_score` clones the estimator on every fold, so it fails loudly if
`__init__` does not follow the convention; and a second `fit` must not extend the history
of the first one.

In [ ]:
print('CV R^2:', cross_val_score(
    LinearRegressionSGD(n_iters=200, random_state=SEED), X_lin, y_lin, cv=5
).round(3))

check = LinearRegressionSGD(n_iters=5, random_state=SEED)
check.fit(x_train, y_train)
check.fit(x_train, y_train)
assert len(check.w_history_) == 6, 'the history must be reset on every fit'
print('OK')

# The full estimator contract (optional, quite strict):
# from sklearn.utils.estimator_checks import check_estimator
# check_estimator(LinearRegressionSGD())

## Here we go again...

Now, instead of `sklearn` Linear Regression we use our own SGD-based implementation.
Note `include_bias=True`: our class has no separate intercept, so the constant feature has to
come from the expansion.

In [ ]:
plt.figure(figsize=(15, 6))
x = np.linspace(-1, 1, 200)
plt.plot(x, true_function(x), c='black', label='true function')
plt.scatter(X_poly, y_poly, marker='*', s=100)
plt.ylim(-2, 2)

for d in set_of_powers:
    model_d = make_pipeline(
        PolynomialFeatures(d, include_bias=True),
        LinearRegressionSGD(lr=1e-2, n_iters=2000, random_state=SEED),
    )
    model_d.fit(X_poly, y_poly)
    plt.plot(x, model_d.predict(x[:, None]), linewidth=2, label="$d=%d$" % d)

plt.legend(loc=1)
plt.title('The same degrees, fitted by SGD')
plt.show()